# 117. 미성형 계량·공급 변화 패턴 전문가
116번에서 본 계량 잔차 방향 차이를 바탕으로 만든 사후 개발가설. 날짜를 전문가나 피처에 넣지 않고 학습 미성형의 6변수 변화 공간을 KMeans 최대2모드로 나눈다. 평가 미성형의 군집은 학습하지 않는다. 정상공정 기계센서6개 전용. 0.5목표 미달을 임계값 사후조정으로 달성했다고 하지 않는다.
특징: 계량시간 dev10/delta1, 계량위치·쿠션·호퍼·평균배압 dev10. 각모드는 RF/ExtraTrees2seed 앙상블. 단일·모드평균·모드최댓값·기존+모드평균을 내부 선택. 가스17 제외/정상566+미성형8.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
d=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d[d['product'].eq('RG3')&(d.PassOrFail.eq(0)|d.Reason.str.contains('미성형',na=False))].reset_index(drop=True)
y=d.PassOrFail.to_numpy(int);g=d.comparison_block.to_numpy();blocks=np.unique(g)
assert len(d)==574 and y.sum()==8
# Shared training-only modes; no date, timestamp or block ID in feature vectors.
cols=['Plasticizing_Time__dev10','Plasticizing_Time__delta1','Plasticizing_Position__dev10','Cushion_Position__dev10','Hopper_Temperature__dev10','Average_Back_Pressure__dev10']
x=d[cols].to_numpy(float)
names=['flat','mode_mean','mode_max','flat_mode_mean']
memo={};audit=[];cache={}
def predict(ref,q):
 assert set(g[ref]).isdisjoint(g[q])
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 if len(np.unique(y[ref]))<2:return np.zeros((len(q),4))
 imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(x[ref[y[ref]==0]])
 tr=imp.transform(x[ref]);te=imp.transform(x[q])
 sc=RobustScaler().fit(tr[y[ref]==0]);z=sc.transform(tr)
 pos=np.flatnonzero(y[ref]==1);normal=np.flatnonzero(y[ref]==0)
 k=min(2,len(np.unique(z[pos],axis=0)))
 cluster=KMeans(n_clusters=k,n_init=10,random_state=42).fit_predict(z[pos]) if k>1 else np.zeros(len(pos),int)
 def fit_score(ids):
  scores=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=2,class_weight='balanced',random_state=seed,n_jobs=2)
    scores.append(m.fit(tr[ids],y[ref][ids]).predict_proba(te)[:,1])
  return np.mean(scores,axis=0)
 flat=fit_score(np.arange(len(ref)));modes=[]
 for mode in np.unique(cluster):
  ids=np.r_[normal,pos[cluster==mode]];modes.append(fit_score(ids))
 mm=np.mean(modes,axis=0);mx=np.max(modes,axis=0)
 out=np.column_stack([flat,mm,mx,(flat+mx)/2]);memo[key]=out
 audit.append(dict(train=len(ref),test=len(q),train_shortshot=len(pos),modes=k,mode_counts=';'.join(str(int((cluster==v).sum())) for v in np.unique(cluster)),overlap=0))
 return out
def threshold(yy,s):
 p,r,c=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)
 k=max(range(len(c)),key=lambda k:(f[k],r[k],c[k]));return float(c[k]),float(f[k])
records=[];choices=[]
for b in blocks:
 ref=np.flatnonzero(g!=b);q=np.flatnonzero(g==b)
 out=predict(ref,q);inner=np.full((len(ref),4),np.nan)
 for ib in np.unique(g[ref]):
  mask=g[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for j,name in enumerate(names):
  cut,fi=threshold(y[ref],inner[:,j]);candidates.append((fi,j,cut))
  for rule,th in [('inner_F1',cut),('normal95',float(np.quantile(inner[y[ref]==0,j],.95,method='higher'))),('normal99',float(np.quantile(inner[y[ref]==0,j],.99,method='higher')))]:
   pr=out[:,j]>=th
   for l,i in enumerate(q):records.append(dict(model=name,rule=rule,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(out[l,j]),threshold=th,prediction=int(pr[l])))
 fi,j,th=max(candidates,key=lambda r:(r[0],-r[1]));pr=out[:,j]>=th
 choices.append(dict(block=b,selected=names[j],threshold=th,inner_F1=fi))
 for l,i in enumerate(q):records.append(dict(model='primary_inner_selected',rule='inner_F1',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(out[l,j]),threshold=th,prediction=int(pr[l])))
 print('Completed recovery mode',b,names[j],flush=True)
p=pd.DataFrame(records);rows=[]
for (name,rule),z in p.groupby(['model','rule']):
 assert len(z)==574 and z.TimeStamp.is_unique
 tn,fp,fn,tp=confusion_matrix(z.actual,z.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,rule=rule,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(z.actual,z.score)))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(audit),'training_audit'),(pd.DataFrame(choices),'choices')]:
 obj.to_csv(t/('rg3_shortshot_recovery_modes_'+name+'.csv'),index=False,encoding='utf-8-sig')
print(m.sort_values('F1',ascending=False).to_string(index=False))


Completed recovery mode 20201021_001626_RG3 mode_mean


Completed recovery mode 20201022_002405_RG3 mode_mean


Completed recovery mode 20201022_044026_RG3 flat


Completed recovery mode 20201023_010128_RG3 flat


Completed recovery mode 20201023_044148_RG3 flat


                 model     rule  TP  FP  FN  TN       F1       AP
             mode_mean inner_F1   6 470   2  96 0.024793 0.015437
              mode_max inner_F1   6 473   2  93 0.024641 0.016063
             mode_mean normal95   1  82   7 484 0.021978 0.015437
                  flat inner_F1   3 279   5 287 0.020690 0.015233
        flat_mode_mean inner_F1   4 381   4 185 0.020356 0.015440
primary_inner_selected inner_F1   2 264   6 302 0.014599 0.016832
        flat_mode_mean normal95   0  58   8 508 0.000000 0.015440
                  flat normal95   0  47   8 519 0.000000 0.015233
                  flat normal99   0  23   8 543 0.000000 0.015233
              mode_max normal99   0  11   8 555 0.000000 0.016063
              mode_max normal95   0  65   8 501 0.000000 0.016063
        flat_mode_mean normal99   0  16   8 550 0.000000 0.015440
             mode_mean normal99   0  24   8 542 0.000000 0.015437
